## Code For Zeroshot


In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import time

# ✅ Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
image_folder = "./RadSpineXR/Test/Annotated_150_images"
csv_path = "./RadSpineXR/Test/sample_150.csv"
output_csv = "./RadSpineXR/Test/BLIP2_zeroshot.csv"

# ✅ Create output directory if needed
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# ✅ Load processor and model
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16,
    device_map="auto"  # ✅ auto-distribution across GPUs
)


# ✅ Load the QA data
df = pd.read_csv(csv_path)
results = []

start = time.time()

# ✅ Process each image + question
for idx, row in df.iterrows():
    image_name = row["image_id"]
    question = row["question"]
    ground_truth = row.get("answer", "")

    image_path = os.path.join(image_folder, image_name)

    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        print(f"⚠️ Error loading image {image_name}: {e}")
        continue

    try:
        # Preprocess and generate
        inputs = processor(images=image, text=question, return_tensors="pt").to(device, torch.float16)
        generated_ids = model.generate(**inputs)
        predicted_answer = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    except Exception as e:
        print(f"⚠️ Error during inference on image {image_name}: {e}")
        predicted_answer = "[ERROR]"

    # Save result
    results.append({
        "image_name": image_name,
        "question": question,
        "generated_answer": predicted_answer,
        "ground_truth": ground_truth
    })

    # Logging
    if idx % 10 == 0:
        print(f"[{idx}] Q: {question}\nA: {predicted_answer}\n")

    # Optional memory cleanup
    if torch.cuda.is_available() and idx % 10 == 0:
        torch.cuda.empty_cache()

    # Save partial progress every 50 rows
    #if idx % 50 == 0 and idx > 0:
       # partial_path = output_csv.replace(".csv", f"_partial_{idx}.csv")
        #pd.DataFrame(results).to_csv(partial_path, index=False)

# ✅ Save final results
output_df = pd.DataFrame(results)
output_df.to_csv(output_csv, index=False)

print(f"\n✅ Results saved to: {output_csv}")
print(f"🕒 Total time: {round(time.time() - start, 2)} seconds")


## Evaluation Metrics Accuracy, BLEU Score, Rouge

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

# Load your CSV file
df = pd.read_csv("./RadSpineXR/Test/BLIP2_zeroshot.csv")  # Replace with actual path if needed

# Clean and prepare
gen_answers = df["generated_answer"].astype(str).str.lower().str.strip()
gt_answers = df["ground_truth"].astype(str).str.lower().str.strip()

# Exact Match Accuracy
exact_matches = (gen_answers == gt_answers).sum()
accuracy = exact_matches / len(df)

# BLEU Score
smoothie = SmoothingFunction().method4
bleu_scores = [sentence_bleu([gt.split()], pred.split(), smoothing_function=smoothie) for gt, pred in zip(gt_answers, gen_answers)]
avg_bleu = sum(bleu_scores) / len(bleu_scores)

# ROUGE Score
rouge = Rouge()
rouge_scores = [rouge.get_scores(pred, gt)[0]['rouge-l']['f'] for gt, pred in zip(gt_answers, gen_answers)]
avg_rouge = sum(rouge_scores) / len(rouge_scores)

# Output
print(f"Accuracy: {accuracy:.2f}")
print(f"Average BLEU: {avg_bleu:.2f}")
print(f"Average ROUGE-L F1: {avg_rouge:.2f}")